<a href="https://colab.research.google.com/github/kasturikirankumar1101-lab/AI_TOOLS/blob/main/PromptGeneratorTool.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#The below code would generate the prompt for chatgpt based on your request.

import gradio as gr
from openai import OpenAI
from google.colab import userdata

# -----------------------------
# 🔐 API Key
# -----------------------------
openai_api_key = userdata.get('OPENAI_API_KEY')

if openai_api_key and openai_api_key.startswith("sk-"):
    print("✅ OpenAI key looks good")
else:
    print("❌ OpenAI key missing")

client = OpenAI(api_key=openai_api_key)

# -----------------------------
# 🧠 System Prompt
# -----------------------------
system_prompt = """
You are a senior-level prompt engineer responsible for high-precision prompt refinement.

Objective:
Refine the given prompt strictly according to the exact requirement and any additional constraints derived from prior user interactions.

Process:
- Deeply analyze the original prompt
- Review all previous user responses for hidden requirements
- Identify missing constraints, ambiguities, redundancies
- Apply only necessary modifications

Rules:
- No assumptions
- No feature expansion
- No vague wording

Output:
- Final Refined Prompt
- Change Log
"""

# -----------------------------
# 🚀 Model Call (Streaming)
# -----------------------------
def call_model(messages):
    stream = client.responses.create(
        model="gpt-4.1-mini",
        input=messages,
        stream=True
    )

    response = ""
    for chunk in stream:
        if chunk.type == "response.output_text.delta":
            response += chunk.delta or ""
            yield response   # ✅ streaming output


# -----------------------------
# 🧩 Build Messages
# -----------------------------
def build_System_prompt(user_message):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]


# -----------------------------
# 🎯 Main Function
# -----------------------------
def generate_prompt(user_msg):
    messages = build_System_prompt(user_msg)

    for chunk in call_model(messages):
        yield chunk   # ✅ THIS FIXES YOUR ISSUE


# -----------------------------
# 🎨 UI
# -----------------------------
with gr.Blocks() as demo:
    gr.Markdown("## 🎨 Required Prompt Generator")

    text_input = gr.Textbox(lines=3, label="Enter Prompt Requirement")
    output = gr.Textbox(lines=20, label="Requested Prompt")

    run_btn = gr.Button("Generate Prompt")

    run_btn.click(
        fn=generate_prompt,
        inputs=text_input,   # ✅ FIXED
        outputs=output
    )

# -----------------------------
# 🚀 Launch
# -----------------------------
demo.launch(debug=True)